In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantum Portfolio Optimizer: A Qiskit Function by Global Data Quantum


> **Note:** Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, On-Prem(IBM Quantum Platform API 경유) Plan 사용자에게만 제공되는 실험적 기능입니다. 현재 미리 보기 릴리스 상태이며 변경될 수 있습니다.

# 개요
Quantum Portfolio Optimizer는 동적 포트폴리오 최적화 문제를 해결하는 Qiskit Function입니다. 이 문제는 수익을 극대화하고 위험을 최소화하기 위해 여러 자산에 걸친 정기적 투자를 재조정하는 금융 분야의 표준 문제입니다. 최첨단 양자 최적화 기법을 적용하여, 양자 컴퓨팅에 대한 전문 지식이 없는 사용자도 최적의 투자 경로를 찾는 데 있어 양자 컴퓨팅의 장점을 활용할 수 있도록 프로세스를 간소화합니다. 포트폴리오 매니저, 계량금융 연구자, 개인 투자자 모두에게 적합하며, 포트폴리오 최적화의 트레이딩 전략 백테스팅을 지원합니다.

# 함수 설명
Quantum Portfolio Optimizer 함수는 Variational Quantum Eigensolver(VQE) 알고리즘을 사용하여 QUBO(Quadratic Unconstrained Binary Optimization) 문제를 풀고, 동적 포트폴리오 최적화 문제를 해결합니다. 사용자는 자산 가격 데이터를 제공하고 투자 제약 조건을 정의하기만 하면, 함수가 양자 최적화 프로세스를 실행하여 최적화된 투자 경로 집합을 반환합니다.

프로세스는 네 가지 주요 단계로 구성됩니다. 먼저 입력 데이터를 양자 호환 문제로 매핑하여 동적 포트폴리오 최적화 문제의 QUBO를 구성하고 양자 연산자(Ising Hamiltonian)로 변환합니다. 다음으로 입력 문제와 VQE 알고리즘이 양자 하드웨어에서 실행될 수 있도록 조정됩니다. 이후 VQE 알고리즘이 양자 하드웨어에서 실행되고, 마지막으로 결과를 후처리하여 최적의 투자 경로를 제공합니다. 또한 출력 품질을 극대화하기 위해 노이즈 인식([SQD](/guides/qiskit-addons-sqd) 기반) 후처리가 포함되어 있습니다.

이 Qiskit Function은 Global Data Quantum의 [출판된 논문](https://arxiv.org/abs/2412.19150)을 기반으로 합니다.
![함수 워크플로우 시각화](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)

# 입력
함수의 입력 인수는 다음 표에 설명되어 있습니다. 자산 데이터 및 기타 문제 사양은 반드시 제공해야 하며, VQE 설정을 추가하여 최적화 프로세스를 사용자 지정할 수 있습니다.

| **이름**               | **타입**           | **설명**                                          | **필수 여부** | **기본값**                          | **예시**                              |
|------------------------|--------------------|----------------------------------------------------------|--------------|--------------------------------------|------------------------------------------|
| assets                 | json               | 자산 가격이 담긴 딕셔너리                         | 예          | -                                    | -                                        |
| qubo_settings          | json               | QUBO 설정                                     | 예          | -                                    | 아래 표의 예시를 참고하세요     |
| ansatz_settings        | json               | ansatz 설정                                   | 아니요           | `None`                                | 아래 표의 예시를 참고하세요.     |
| optimizer_settings     | json               | 옵티마이저 설정                                | 아니요           | `None`                                | 아래 표의 예시를 참고하세요.     |
| backend                | str                | QPU Backend 이름                                     | 아니요           |  -    | `"ibm_torino"`                           |
| previous_session_id    | list of str        | 이전 실행에서 데이터를 가져올 Session ID 목록[(*)](#1-note) | 아니요           | 빈 목록                           | `["session_id_1", "session_id_2"]`      |
| apply_postprocess      | bool               | 노이즈 인식 SQD 후처리 적용                      | 아니요           | `True`                                | `True`                                   |
| tags                   | list of strings    | 실험을 식별하기 위한 태그 목록                  | 아니요           | 빈 목록                           | `["optimization", "quantum_computing"]`  |

<span id="1-note"></span>
*이전 Session에서 처리된 작업을 재개하거나 가져오려면, `previous_session_id` 매개변수에 Session ID 목록을 전달해야 합니다. 이는 최적화 작업이 프로세스 중 오류로 인해 완료되지 못한 경우 특히 유용합니다. 이를 위해 초기 실행 시 사용했던 동일한 인수와 함께 설명된 대로 `previous_session_id` 목록을 제공해야 합니다.

> **Caution:** 이전 Session 데이터를 로드하는 데(최적화 재개 목적) 최대 1시간이 소요될 수 있습니다.

## `assets`
데이터는 특정 날짜의 금융 자산 종가 정보를 저장하는 JSON 객체로 구성해야 합니다. 형식은 다음과 같습니다.

- 1차 키(문자열): 금융 자산의 이름 또는 티커 심볼(예: "8801.T").
- 2차 키(문자열): YYYY-MM-DD 형식의 날짜.
- 값(숫자): 해당 날짜의 자산 종가. 정규화 또는 비정규화 가격 모두 입력 가능합니다.

*모든 딕셔너리는 동일한 2차 키(날짜)를 가져야 합니다. 특정 자산에 다른 자산의 날짜가 누락된 경우, 일관성을 위해 데이터를 채워야 합니다. 예를 들어, 해당 자산의 마지막으로 추적된 종가를 사용하여 채울 수 있습니다.*

### 예시
``` py
{
    "8801.T": {
        "2023-01-01": 2374.0,
        "2023-01-02": 2374.0,
        "2023-01-03": 2374.0,
        "2023-01-04": 2356.5,
        ...
    },
    "AAPL": {
        "2023-01-01": 145.2,
        "2023-01-02": 146.5,
        "2023-01-03": 147.3,
        "2023-01-04": 148.1,
        ...
    },
    ...
}
```

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

> **Note:** 자산 데이터에는 최소한 ``(nt+1) * dt``([`qubo_settings`](#qubo_settings) 입력 섹션 참고) 타임스탬프(예: 일 단위)의 종가가 포함되어야 합니다.

## `qubo_settings`
다음 표는 `qubo_settings` 딕셔너리의 키를 설명합니다. 시간 스텝 수 `nt`, 해상도 Qubit 수 `nq`, `max_investment`를 지정하거나 다른 기본값을 변경하여 딕셔너리를 구성하세요.

| 이름                | 타입    | 설명                                                                 | 필수 여부 | 기본값 | 예시 |
|---------------------|---------|-----------------------------------------------------------------------------|----------|---------|---------|
| nt                  | int     | 시간 스텝 수                                                         | 예      | -       | 4       |
| nq                  | int     | 해상도 Qubit 수                                                                | 예      | -       | 4       |
| max_investment      | float   | 모든 자산에 걸쳐 투자되는 최대 통화 단위 수                           | 예      | -       | 10      |
| dt*                  | int     | 각 시간 스텝에서 고려되는 시간 윈도우. 단위는 자산 데이터의 키 사이 시간 간격과 일치합니다                                                 | 아니요       | 30      | -       |
| risk_aversion       | float     | 위험 회피 계수                                   | 아니요       | 1000    | -       |
| transaction_fee     | float     | 거래 수수료 계수                                                 | 아니요       | 0.01   | -       |
| restriction_coeff   | float   | QUBO 공식 내에서 문제 제약 조건을 적용하기 위한 라그랑주 승수 | 아니요       | 1       | -       |

## `ansatz_settings`
기본 옵션을 수정하려면 `ansatz_settings` 매개변수에 다음 키가 포함된 딕셔너리를 생성하세요. 기본적으로 ansatz는 `"real_amplitudes"`로 설정되며, 추가 옵션(다음 표 참고)은 모두 `False`로 설정됩니다.

| 이름                  | 타입     | 설명                                                                 | 필수 여부 | 기본값 |
|-----------------------|----------|-----------------------------------------------------------------------------|----------|---------|
| ansatz[*](#single-star)                | str      | 사용할 ansatz                                             | 아니요      | `"real_amplitudes"` |
| multiple_passmanager[**](#double-star)  | bool     | 다중 passmanager 서브루틴 활성화(Tailored ansatz에는 사용 불가) | 아니요       | `False`   |
| dd_enable   | bool     | 동적 디커플링 추가                                    | 아니요       | `False`   |

<span id="single-star"></span>
\* 사용 가능한 ansatz 목록
- `real_amplitudes`
- `cyclic`
- `optimized_real_amplitudes`
- `tailored` (`ibm_torino` Backend, 자산 7개, 시간 스텝 4개, 해상도 Qubit 4개에만 사용 가능)

<span id="double-star"></span>
** ``multiple_passmanager``가 ``False``로 설정된 경우, 함수는 `optimization_level=3`의 기본 Qiskit pass manager를 사용합니다. ``True``로 설정하면, ``multiple_passmanager`` 서브루틴이 세 가지 pass manager를 비교합니다: 기존 기본 Qiskit pass manager, QPU 인접 체인 위에 Qubit을 매핑하는 pass manager, 그리고 [AI Transpiler](/guides/ai-transpiler-passes) 서비스. 그런 다음 추정 누적 오류가 가장 낮은 pass manager가 선택됩니다.

## `optimizer_settings`
이 매개변수는 최적화 프로세스의 일부 조정 가능한 옵션이 담긴 딕셔너리입니다.

| 이름               | 타입   | 설명                                     | 필수 여부 | 기본값               |
|--------------------|--------|-------------------------------------------------|----------|-----------------------|
| primitive_options  | json   | primitive 설정               | 아니요      | -                     |
| optimizer          | str    | 선택된 고전 옵티마이저                            | 아니요       | `"differential_evolution"` |
| optimizer_options  | json   | 옵티마이저 구성                  | 아니요       | -                     |

> **Note:** 현재 사용 가능한 옵티마이저 옵션은 `"differential_evolution"`뿐입니다.

`primitive_options` 및 `optimizer_options` 키 아래에 다음 매개변수를 가진 딕셔너리를 설정합니다.

### `primitive_options`
| 이름              | 타입   | 설명                                | 필수 여부 | 기본값                    | 예시                    |
|-------------------|--------|--------------------------------------------|----------|----------------------------|----------------------------|
| sampler_shots     | int    | Sampler의 샷 수.            | 아니요       | 100000                     | -                          |
| estimator_shots   | int    | Estimator의 샷 수.          | 아니요       | 25000                      | -                          |
| estimator_precision | float | 기댓값의 목표 정밀도. 지정된 경우 `estimator_shots` 대신 정밀도가 사용됩니다. | 아니요 | `None` | 0.015625 · (1 / sqrt(4096)) |
| max_time          | int or str | 런타임 Session이 강제 종료되기 전 열려 있을 수 있는 최대 시간. 초(int) 또는 `"2h 30m 40s"`와 같은 문자열로 지정할 수 있습니다. 시스템이 지정한 최댓값보다 작아야 합니다. | 아니요 | `None` | `"1h 15m"`                |

### `optimizer_options`
| 이름              | 타입     | 설명                              | 필수 여부 | 기본값       |
|-------------------|----------|------------------------------------------|----------|---------------|
| num_generations   | int      | 세대 수                    | 아니요       | 20            |
| population_size   | int      | 개체군 크기                    | 아니요       | 20            |
| mutation_range    | list   | 최대 및 최소 돌연변이 인수              | 아니요       | [0, 0.25]     |
| recombination     | float      | 재조합 인수                     | 아니요       | 0.4           |
| max_parallel_jobs | int      | 병렬 실행되는 최대 QPU 작업 수  | 아니요       | 3             |
| max_batchsize | int      | 최대 배치 크기  | 아니요       | 200     |

> **Note:** - 차등 진화에 의해 평가되는 세대 수는 초기 개체군이 포함되므로 `num_generations` + 1입니다.
> - 총 Circuit 수는 `(num_generations + 1) * population_size`로 계산됩니다.
> - 개체군 크기를 크게 하고 세대 수를 늘리면 일반적으로 최적화 결과의 품질이 향상됩니다. 그러나 개체군 크기 120, 세대 수 20(예: ``120 * 21 = 2520``개의 총 Circuit)을 초과하는 것은 권장하지 않습니다. 이 경우 과도한 수의 Circuit이 생성되어 처리에 많은 계산 비용과 시간이 소요될 수 있습니다.
> 
> - 함수는 이전 최적화 재개를 지원하며, 세대 수를 늘리는 것은 항상 가능합니다(`previous_session_id`를 제외한 동일한 입력과 늘어난 ``num_generations``를 제공하면 됩니다).

> **Note:** Qiskit Runtime 작업 한도를 준수하세요.
> 
> - Sampler: `sampler_shots <= 10_000_000`.
> - Estimator: `max_batchsize * estimator_shots * observable_size <= 10_000_000` (이 함수의 경우 observable의 모든 항이 교환 가능하므로 `observable_size=1`).
> 
> 자세한 내용은 [작업 한도](/guides/job-limits) 가이드를 참고하세요.

# 출력
함수는 두 개의 딕셔너리를 반환합니다: 최적의 솔루션과 관련 최소 목적 비용을 포함한 최상의 최적화 결과가 담긴 `"result"` 딕셔너리, 그리고 최적화 프로세스 중 얻어진 모든 결과의 데이터와 각각의 메트릭이 담긴 `"metadata"` 딕셔너리입니다.

첫 번째 딕셔너리는 최고 성능 솔루션에 초점을 맞추고, 두 번째 딕셔너리는 목적 비용 및 기타 관련 메트릭을 포함한 모든 솔루션에 대한 상세 정보를 제공합니다.

## 출력 딕셔너리:
| **이름**    | **타입**                     | **설명**                                                                 | **예시**  |
|-------------|------------------------------|---------------------------------------------------------------------------------|--------------|
| result      | dict[str, dict[str, float]]  | 시간에 따른 투자 전략으로, 각 타임스탬프가 자산별 투자 비중에 매핑됩니다(각 비중은 총 투자 금액으로 정규화된 투자 금액입니다). | `{'time_1': {'asset_1': 0.2, 'asset_2': 0.3, ...}, ...}` |
| metadata    | dict[str, Any]               | 솔루션, 비용, 메트릭 등 분석 중 생성된 데이터.    | 아래 예시를 참고하세요 |

### `metadata` 딕셔너리 설명
| **이름**                 | **타입**              | **설명**                                                                                     | **예시**                  |
|--------------------------|-----------------------|-----------------------------------------------------------------------------------------------------|------------------------------|
| session_id               | str                   | IBM Quantum Session의 고유 식별자.                          | `"d0h30qjvpqf00084fgw0"`        |
| all_samples_metrics | dict                 | 비용 또는 제약 조건 등 각 후처리된 샘플에 대한 다양한 메트릭이 담긴 딕셔너리. | 아래 설명을 참고하세요        |
| sampler_counts           | dict[str, int]        | 키가 샘플링된 솔루션의 비트열 표현이고 값이 해당 카운트인 딕셔너리. | `{"101010": 3, "111000": 1}` |
| asset_order              | list[str]             | 투자 전략 내 각 시간 스텝에서 자산의 투자 순서가 담긴 목록. | `["Asset_0", "Asset_1", "Asset_3"]`     |
| QUBO                     | list[list[float]]     | 문제의 QUBO 행렬.                                                                         | `[[-6.96e-01, 5.81e-01, -1.26e-02, 0.00e+00], ...]`     |
| resource_summary           | dict[str, dict[str, float]] | 프로세스의 각 단계에 걸친 CPU 및 QPU 사용 시간(초 단위) 요약.                | `{'RUNNING: EXECUTING_QPU': {'CPU_TIME': 412.84, 'QPU_TIME': 87.22}, ...}` |

### `all_samples_metrics` 딕셔너리 설명
| **이름**                | **타입**       | **설명**                                                                                                  | **예시**                  |
|-------------------------|----------------|------------------------------------------------------------------------------------------------------------------|------------------------------|
| investment_trajectories | list[list]     | 디코딩된 양자 상태에서 도출된 투자 전략. | `[[1, 2, 2], [1, 2, 1]]`     |

| counts                  | list[int]      | 각 투자 경로가 샘플링된 횟수. 인덱스는 `investment_trajectories`와 일치합니다.                | `[5, 3]`                     |
| objective_costs         | list[float]    | 각 투자 경로에 대한 목적 함수 값으로, 낮은 순에서 높은 순으로 정렬됩니다.                 | `[0.98, 1.25]`               |
| sharpe_ratios           | list[float]    | 각 투자 경로에 대한 위험 조정 성과(샤프 비율). 인덱스 기준으로 정렬됩니다.                      | `[1.1, 0.7]`                 |
| returns                 | list[float]    | 각 투자 경로에 대한 기대 수익. 인덱스 기준으로 정렬됩니다.                                               | `[0.15, 0.10]`               |
| rest_breaches           | list[float]    | 각 투자 경로 내 최대 제약 조건 이탈 값. 인덱스 기준으로 정렬됩니다.                               | `[0.0, 0.25]`                |
| transaction_costs       | list[float]    | 각 투자 경로와 관련된 추정 거래 비용. 인덱스 기준으로 정렬됩니다.                        | `[0.01, 0.02]`               |

# 시작하기
[API 키](https://quantum.cloud.ibm.com/)를 사용하여 인증하고 다음과 같이 Qiskit Function을 선택하세요. (이 코드 조각은 이미 [계정을 로컬 환경에 저장](/guides/functions#install-qiskit-functions-catalog-client)했다고 가정합니다.)

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

## 예제: 7개 자산을 이용한 동적 포트폴리오 최적화
이 예제에서는 동적 포트폴리오 최적화(DPO) 함수를 실행하고 최적의 성능을 위해 설정을 조정하는 방법을 보여줍니다. 원하는 결과를 달성하기 위해 파라미터를 미세 조정하는 자세한 단계를 포함합니다.

이 경우 7개의 자산, 4개의 시간 단계, 4개의 해상도 Qubit를 포함하며, 총 112개의 Qubit가 필요합니다.
### 1. 포트폴리오에 포함된 자산 읽기
포트폴리오의 모든 자산이 특정 경로의 폴더에 저장되어 있다면, 다음 함수를 사용하여 `pandas.DataFrame`으로 로드하고 `dict` 형식 객체로 변환할 수 있습니다.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

이 예제에서는 [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y), [XS2239553048](https://finance.yahoo.com/quote/XS2239553048) 자산을 사용했습니다. 다음 그림은 이 예제에서 사용된 데이터를 보여주며, 2023년 1월 1일부터 9월 1일까지 자산의 일일 종가 변동을 나타냅니다.

이 예제에서는 날짜 간 균일성을 보장하기 위해 비거래일에는 이전 거래일의 종가를 채웠습니다. 선택된 자산이 거래일이 다른 여러 시장에서 왔기 때문에 데이터셋의 일관성을 위해 표준화하는 이 단계가 필수적입니다.
![자산의 과거 데이터 시각화](../docs/images/guides/global-data-quantum-optimizer/assets.avif)

### 2. 문제 정의

`qubo_settings` 딕셔너리에서 파라미터를 구성하여 문제의 사양을 정의합니다.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 3. 옵티마이저 및 ansatz 설정 정의 (선택 사항)
선택적으로 옵티마이저와 그 파라미터의 선택, 그리고 primitive와 그 구성의 지정을 포함하여 최적화 프로세스에 대한 특정 요구 사항을 정의할 수 있습니다.

Tailored Ansatz의 경우, 선택된 population size는 이 값이 안정적이고 효율적인 최적화를 제공한다는 이전 실험에 기반합니다.

Real Amplitudes Ansatz의 경우, ``population_size``와 Circuit의 Qubit 수 사이의 선형 관계를 따를 수 있습니다. 대략적인 경험 법칙으로, `real_amplitudes` ansatz에 대해 **최소** ``population_size ~ 0.8 * n_qubits``를 사용하는 것이 권장됩니다.

Optimized Real Amplitudes가 Real Amplitudes ansatz보다 더 나은 최적화 성능을 보일 것으로 예상됩니다. 그러나 이 ansatz에서 최적화할 변수의 수가 Real Amplitudes의 경우보다 훨씬 빠르게 증가합니다([논문](https://arxiv.org/pdf/2412.19150) 참조). 따라서 큰 문제의 경우 Optimized Real Amplitudes는 더 많은 Circuit 실행이 필요합니다. Optimized Real Amplitudes는 최대 100 Qubit가 필요한 문제에 유용할 가능성이 높지만, ``population_size`` 파라미터를 설정할 때 주의하는 것이 좋습니다. ``population_size`` 확장의 예로, 이전 표에서 84-Qubit 문제의 경우 Optimized Real Amplitudes는 120 ``population_size``가 필요하고, 56-Qubit 문제의 경우 ``population_size`` 40이면 충분합니다.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

특정 ansatz를 선택하는 것도 가능합니다. 다음은 ``'Tailored'`` ansatz를 사용합니다.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 4. 문제 실행

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

### 5. 결과 조회
[출력](#output) 섹션에서 언급한 것처럼, 함수는 목적 함수 값에 따라 낮은 것부터 높은 것 순으로 정렬된 투자 궤적이 포함된 딕셔너리를 반환합니다. 이 결과 세트를 통해 가장 낮은 비용의 궤적과 해당 투자 평가를 식별할 수 있습니다. 또한 다양한 궤적을 분석하여 특정 요구 사항이나 목표에 가장 잘 맞는 궤적을 선택할 수 있습니다. 이러한 유연성은 다양한 선호도나 시나리오에 맞게 선택을 조정할 수 있게 합니다.
프로세스 중 발견된 가장 낮은 목적 비용을 달성한 결과 전략을 먼저 확인합니다.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
*   Request access to the function by filling in [this form](https://www.globaldataquantum.com/en/quantum-portfolio-optimizer/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>